# 02 · Yes/No probabilities on Colab

One forward pass per item for each model, reading the next-token probabilities of "Yes" and "No".

1. GPU 4T
2. Gated models (Llama, Gemma) need a Hugging Face token with access; the default list below uses ungated models only.
3. Run once with `SMOKE_TEST = True` (24 items per model) and look at the diagnostics, then set it to `False`.
4. Results go to Google Drive, so they survive a disconnect. Re-running resumes where it stopped. Copy the `logprobs-*.jsonl` files to `data/interim/` in the local repo for notebook 03.

In [2]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/laudima/causal-evaluation.git"

if IN_COLAB:
    ROOT = Path("/content/causal-evaluation")
    if not ROOT.exists():
        !git clone -q {REPO_URL} {ROOT}
    !pip install -q -e "{ROOT}[colab]"
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.8 MB/s eta 0:00:00
  Building editable for causal-evaluation (pyproject.toml) ... done


In [3]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/causal-evaluation/results")
else:
    OUT_DIR = Path(os.environ.get("CE_INTERIM", ROOT / "data" / "interim"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUT_DIR)

Mounted at /content/drive
/content/drive/MyDrive/causal-evaluation/results


## Rebuild the sample and check it matches notebook 01

In [4]:
from causal_evaluation.data.variants import build_variant_sample, write_jsonl

SEED = 20260903
SIZE_PER_VARIANT = 1000
EXPECTED_SHA = "28feb2479c3b6a59e16c9754d6b5e3752c585004da220198cf3cdc6c0ea656d2"

items = build_variant_sample(ROOT / "data" / "raw", SIZE_PER_VARIANT, SEED)
sample_sha = write_jsonl(items, OUT_DIR / "variants-sample.jsonl")
print(f"{len(items):,} items, SHA-256 {sample_sha}")
if EXPECTED_SHA:
    assert sample_sha == EXPECTED_SHA, "Sample differs from the local one"

3,000 items, SHA-256 28feb2479c3b6a59e16c9754d6b5e3752c585004da220198cf3cdc6c0ea656d2


## Models

In [5]:
# from huggingface_hub import login; login()  # only for gated models

# dtype: float32 keeps P(Yes) independent of batching; models that do not fit a T4 (15 GB)
# in float32 load in 4-bit with float16 compute. Report the dtype of each model in the paper.
MODELS = [
    {"name": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "dtype": "float32", "load_in_4bit": False, "batch_size": 8},
    {"name": "Qwen/Qwen2.5-1.5B-Instruct", "dtype": "float32", "load_in_4bit": False, "batch_size": 8},
    {"name": "microsoft/Phi-3.5-mini-instruct", "dtype": "float16", "load_in_4bit": True, "batch_size": 8},
    {"name": "Qwen/Qwen2.5-7B-Instruct", "dtype": "float16", "load_in_4bit": True, "batch_size": 8},
]
SMOKE_TEST = False
SMOKE_ITEMS = 24


def result_path(model_name):
    suffix = "-smoke" if SMOKE_TEST else ""
    return OUT_DIR / f"logprobs-{model_name.split('/')[-1]}{suffix}.jsonl"

## Batch-invariance check (smoke test only)

P(Yes) for an item should not depend on which other items share its batch. Values above about 0.01 mean the precision setting for that model should change.

In [6]:
from causal_evaluation.models.logprob import check_batch_invariance

if SMOKE_TEST:
    probe = [items[0], items[1500], items[2999], items[7]]
    for spec in MODELS:
        diff = check_batch_invariance(spec["name"], probe, spec["load_in_4bit"], spec["dtype"])
        print(f"{spec['name']}: max |batched - alone| = {diff:.4f}" + ("  <-- check dtype" if diff > 0.01 else ""))

## Run

In [7]:
import gc

import torch

from causal_evaluation.models.logprob import run_model

run_items = items[:SMOKE_ITEMS] if SMOKE_TEST else items
for spec in MODELS:
    path = result_path(spec["name"])
    scored = run_model(
        run_items,
        spec["name"],
        path,
        batch_size=spec["batch_size"],
        load_in_4bit=spec["load_in_4bit"],
        dtype=spec["dtype"],
    )
    print(f"{spec['name']}: {scored} new items -> {path.name}")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

HuggingFaceTB/SmolLM2-1.7B-Instruct: 0 new items -> logprobs-SmolLM2-1.7B-Instruct.jsonl
Qwen/Qwen2.5-1.5B-Instruct: 0 new items -> logprobs-Qwen2.5-1.5B-Instruct.jsonl
microsoft/Phi-3.5-mini-instruct: 0 new items -> logprobs-Phi-3.5-mini-instruct.jsonl
Qwen/Qwen2.5-7B-Instruct: 0 new items -> logprobs-Qwen2.5-7B-Instruct.jsonl


## Diagnostics

`answer_mass` is the probability the model puts on Yes/No tokens at all. If its median is low (say below 0.5) the model wants to start with something else (look at `top_token`) and P(Yes) is less meaningful for that model.

In [8]:
import pandas as pd

from causal_evaluation.evaluation.variant_analysis import load_scored, rate_table

paths = [p for p in sorted(OUT_DIR.glob("logprobs-*.jsonl")) if p.name.endswith("-smoke.jsonl") == SMOKE_TEST]
scored = load_scored(OUT_DIR / "variants-sample.jsonl", paths)
display(scored.groupby("model").agg(items=("item_id", "size"), median_answer_mass=("answer_mass", "median"), yes_rate=("pred", lambda s: (s == "yes").mean())))
display(scored.groupby("model")["top_token"].agg(lambda s: s.value_counts().head(3).to_dict()))
rate_table(scored, ["model", "variant"])

,items,median_answer_mass,yes_rate
model,,,
HuggingFaceTB/SmolLM2-1.7B-Instruct,3000,0.991181,0.000667
Qwen/Qwen2.5-1.5B-Instruct,3000,0.999718,0.497333
Qwen/Qwen2.5-7B-Instruct,3000,1.000000,0.479000
microsoft/Phi-3.5-mini-instruct,3000,0.999999,0.327333


,top_token
model,
HuggingFaceTB/SmolLM2-1.7B-Instruct,"{'No': 2998, 'Yes': 2}"
Qwen/Qwen2.5-1.5B-Instruct,"{'No': 1508, 'Yes': 1492}"
Qwen/Qwen2.5-7B-Instruct,"{'No': 1569, 'Yes': 1431}"
microsoft/Phi-3.5-mini-instruct,"{'No': 2018, 'Yes': 982}"


,model,variant,n,k,rate,lo,hi
0,HuggingFaceTB/SmolLM2-1.7B-Instruct,anticommonsense,1000,500,0.500,0.469070,0.530930
1,HuggingFaceTB/SmolLM2-1.7B-Instruct,commonsense,1000,502,0.502,0.471062,0.532922
2,HuggingFaceTB/SmolLM2-1.7B-Instruct,noncommonsense,1000,500,0.500,0.469070,0.530930
3,Qwen/Qwen2.5-1.5B-Instruct,anticommonsense,1000,501,0.501,0.470066,0.531927
4,Qwen/Qwen2.5-1.5B-Instruct,commonsense,1000,515,0.515,0.484026,0.545859
5,Qwen/Qwen2.5-1.5B-Instruct,noncommonsense,1000,534,0.534,0.503011,0.564729
6,Qwen/Qwen2.5-7B-Instruct,anticommonsense,1000,596,0.596,0.565275,0.625990
7,Qwen/Qwen2.5-7B-Instruct,commonsense,1000,571,0.571,0.540110,0.601346
8,Qwen/Qwen2.5-7B-Instruct,noncommonsense,1000,556,0.556,0.525049,0.586522
9,microsoft/Phi-3.5-mini-instruct,anticommonsense,1000,572,0.572,0.541115,0.602334
